In [1]:
import pandas as pd
import numpy as np
import joblib

from sqlalchemy import create_engine, inspect
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
DB_path = "data/phishing.db"
Table = "phishing.db"


Target = "label"
Numerical_cols = ['LineOfCode',
                'LargestLineLength',
                'NoOfURLRedirect',
                'NoOfSelfRedirect', 
                'NoOfPopup', 
                'NoOfiFrame', 
                'NoOfImage',
                'NoOfSelfRef',
                'NoOfExternalRef', 
                'Robots', 
                'IsResponsive',
                'DomainAgeMonths']
Categorical_cols = ['Industry',
                    'HostingProvider']
Numerical_cols = Numerical_cols + ["LineOfCode_missing"]

Numerical_Imputer_Strategy = "median"
Categorical_Imputer_Strategy = "most_frequent"

Test_Size = 0.2
Random_state = 42



In [3]:
def load_data():
    engine = create_engine("sqlite:///data/phishing.db")
    df = pd.read_sql_table("phishing_data", engine)
    return df

def load_cleaned_data():
    df = load_data()
    df = df.drop(columns=["Unnamed: 0"])
    df.loc[df["NoOfImage"] < 0, "NoOfImage"] = 0
    return df

df = load_cleaned_data()
X = df.drop(columns=["label"],axis = 1)
y = df["label"]

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
#from config.py import Numerical_Imputer_Strategy, Categorical_Imputer_Strategy

def preprocessor():
    numerical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean = False)),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy ="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown= "ignore")),
    ])

    return ColumnTransformer([
        ("num", numerical_pipe, Numerical_cols),
        ("cat", categorical_pipe, Categorical_cols)
    ])
    

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.dummy import DummyClassifier

def main():
    df = load_cleaned_data()
    X = df.drop(columns=[Target])
    y = df[Target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =0.2, random_state = 42, stratify=y)

    pipe = Pipeline([
        ("preprocessor", preprocessor()),
        ("model", LogisticRegression(max_iter=2000)),
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    print(classification_report(y_test, y_pred))

In [17]:
Target = "label"

def preprocessor():
    numerical_pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("imputer", KNNImputer(n_neighbors = 5, weights = "distance")),
        # critical: with_mean=False for sparse matrices
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numerical_pipe, Numerical_cols),
        ("cat", categorical_pipe, Categorical_cols),
    ])
df = load_cleaned_data()
df["LineOfCode_missing"] = df["LineOfCode"].isna().astype(int)
# sanity: strip whitespace from column names
df.columns = df.columns.astype(str).str.strip()

all_cols = set(df.columns)

num_missing = [c for c in Numerical_cols if c not in all_cols]
cat_missing = [c for c in Categorical_cols if c not in all_cols]
target_missing = (Target not in all_cols)

print("Missing numeric:", num_missing)
print("Missing categorical:", cat_missing)
print("Target missing?", target_missing)

# also helpful: see near-matches (case / spaces)
print("\nDataFrame columns (sample):", sorted(list(all_cols))[:40])


# (optional but recommended) clean category strings
for c in ["Industry", "HostingProvider"]:
    df[c] = df[c].astype(str).str.strip().str.lower().replace({"nan": "unknown", "none": "unknown"})

X = df.drop(columns=[Target])
y = df[Target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor()),
        ("model", LogisticRegression(max_iter=5000, solver="saga", n_jobs=-1,
                                     class_weight="balanced", random_state=42)),
    ]),
    "LinearSVC": Pipeline([
        ("preprocessor", preprocessor()),
        ("model", LinearSVC(C=1.0, class_weight="balanced", random_state=42)),
    ]),
    "ExtraTreesClassifier": Pipeline([
        ("preprocessor", preprocessor_dense()),
        ("model", ExtraTreesClassifier(n_estimators=600, n_jobs=-1,
                                       random_state=42, class_weight="balanced")),
    ]),
}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print("\n" + "="*20, name, "="*20)
    print(classification_report(y_test, pred))






Missing numeric: []
Missing categorical: []
Target missing? False

DataFrame columns (sample): ['DomainAgeMonths', 'HostingProvider', 'Industry', 'IsResponsive', 'LargestLineLength', 'LineOfCode', 'LineOfCode_missing', 'NoOfExternalRef', 'NoOfImage', 'NoOfPopup', 'NoOfSelfRedirect', 'NoOfSelfRef', 'NoOfURLRedirect', 'NoOfiFrame', 'Robots', 'label']

==================== Logistic Regression ====================
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       944
           1       0.86      0.84      0.85      1156

    accuracy                           0.83      2100
   macro avg       0.83      0.83      0.83      2100
weighted avg       0.83      0.83      0.83      2100



C:\Users\ryans\anaconda3\envs\stablediffusion\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\ryans\anaconda3\envs\stablediffusion\Lib\site-packages\sklearn\svm\_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



==================== LinearSVC ====================
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       944
           1       0.86      0.84      0.85      1156

    accuracy                           0.83      2100
   macro avg       0.83      0.83      0.83      2100
weighted avg       0.83      0.83      0.83      2100


==================== ExtraTreesClassifier ====================
              precision    recall  f1-score   support

           0       0.83      0.76      0.80       944
           1       0.82      0.88      0.85      1156

    accuracy                           0.82      2100
   macro avg       0.83      0.82      0.82      2100
weighted avg       0.82      0.82      0.82      2100



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def preprocessor():
    numerical_pipe = Pipeline([
        ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numerical_pipe, Numerical_cols),
            ("cat", categorical_pipe, Categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import ExtraTreesClassifier

def preprocessor_dense():
    numerical_pipe = Pipeline([
        ("scaler", StandardScaler()),  # dense now, so normal scaler is fine
        ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("num", numerical_pipe, Numerical_cols),
        ("cat", categorical_pipe, Categorical_cols),
    ])


extratrees_pipe.fit(X_train, y_train)
y_pred = extratrees_pipe.predict(X_test)
print("ExtraTrees\n", classification_report(y_test, y_pred))


NameError: name 'extratrees_pipe' is not defined

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

def get_models(random_state: int):
    """6 popular tabular classifiers (sklearn-only)."""
    return {
        "logreg": LogisticRegression(max_iter=5000, solver="saga"),
        "linearsvc": LinearSVC(),  # strong for many features; no predict_proba
        "random_forest": RandomForestClassifier(
            n_estimators=400, random_state=random_state, n_jobs=-1
        )
    }

